# Experiment: DTSE Chapter 8 Frozen Artifact Repro Checks

This notebook verifies Chapter 8 artifact integrity and table-level reproducibility from the frozen DTSE bundle.

**Scope:** artifact integrity, aggregation consistency, and signal-table reproducibility.

**Non-goal:** real-world predictive validation.


In [1]:
from pathlib import Path
import json
import hashlib
import pandas as pd

ROOT = Path('/Users/devinsonpena/Desktop/Files/cas-depin-tokenomics')
EXPORTS = ROOT / 'exports' / 'dtse'
MANIFEST_PATH = EXPORTS / 'run_manifest_2026-02-24.json'

TABLES = {
    'weekly': EXPORTS / 'tables' / 'dtse_weekly_summary.csv',
    'final': EXPORTS / 'tables' / 'dtse_final_summary.csv',
    'signals': EXPORTS / 'tables' / 'dtse_cross_scenario_signals_ono.csv',
    'synthesis': EXPORTS / 'tables' / 'dtse_cross_scenario_synthesis.csv',
    'thresholds': EXPORTS / 'tables' / 'dtse_threshold_registry.csv',
}

with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

manifest_summary = {
    'scenario_grid_id': manifest['scenario_grid_id'],
    'commit_hash': manifest['dtse_repo']['commit_hash'],
    'master_seed': manifest['master_seed'],
    'n_sims': manifest['n_sims'],
    'horizon_weeks': manifest['horizon_weeks'],
    'artifact_count': manifest['artifact_count'],
}
manifest_summary


{'scenario_grid_id': 'dtse_ch8_grid_v1',
 'commit_hash': '0f7b69d0b5776137bddcea6ed9fc642259764792',
 'master_seed': 20260224,
 'n_sims': 60,
 'horizon_weeks': 52,
 'artifact_count': 18}

## 1) Manifest Integrity
Checks that each listed artifact exists and matches its SHA256 in the run manifest.


In [2]:
missing = []
hash_mismatch = []

for a in manifest['artifacts']:
    path = ROOT / a['path']
    if not path.exists():
        missing.append(a['path'])
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != a['sha256']:
        hash_mismatch.append({'path': a['path'], 'expected': a['sha256'], 'observed': digest})

integrity_report = {
    'artifacts_listed': len(manifest['artifacts']),
    'missing_count': len(missing),
    'hash_mismatch_count': len(hash_mismatch),
}

integrity_report


{'artifacts_listed': 18, 'missing_count': 0, 'hash_mismatch_count': 0}

## 2) Table Load + Cardinality


In [3]:
weekly = pd.read_csv(TABLES['weekly'])
final = pd.read_csv(TABLES['final'])
signals = pd.read_csv(TABLES['signals'])
synthesis = pd.read_csv(TABLES['synthesis'])
thresholds = pd.read_csv(TABLES['thresholds'])

{
    'weekly_rows': len(weekly),
    'final_rows': len(final),
    'signals_rows': len(signals),
    'synthesis_rows': len(synthesis),
    'threshold_rows': len(thresholds),
}


{'weekly_rows': 13000,
 'final_rows': 25,
 'signals_rows': 4,
 'synthesis_rows': 4,
 'threshold_rows': 21}

## 3) Final Summary Consistency Check
Recompute final-week medians from `weekly` and compare against `final`.


In [4]:
med = weekly.pivot_table(
    index=['scenario_id', 'profile_id', 'week'],
    columns='metric',
    values='median',
    aggfunc='first'
).reset_index()

week51 = med[med['week'] == 51].copy()
week0_providers = med[med['week'] == 0][['scenario_id', 'profile_id', 'providers']].rename(columns={'providers': 'providers_w0'})
calc = week51.merge(week0_providers, on=['scenario_id', 'profile_id'], how='left')

calc['retention_final_calc'] = calc['providers'] / calc['providers_w0'].replace(0, pd.NA)
calc['burn_to_mint_final_calc'] = calc['burned'] / calc['minted'].replace(0, pd.NA)

joined = final.merge(calc, on=['scenario_id', 'profile_id'], how='left')

pairs = [
    ('providers_final', 'providers'),
    ('solvency_final', 'solvencyScore'),
    ('profit_final', 'profit'),
    ('utilisation_final', 'utilisation'),
    ('price_final', 'price'),
    ('churn_final', 'churnCount'),
    ('capacity_final', 'capacity'),
    ('incentive_final', 'incentive'),
    ('retention_final', 'retention_final_calc'),
    ('burn_to_mint_final', 'burn_to_mint_final_calc'),
]

max_abs_diffs = {}
for left, right in pairs:
    max_abs_diffs[f'{left}~{right}'] = float((joined[left] - joined[right]).abs().max())

max_abs_diffs


{'providers_final~providers': 0.0,
 'solvency_final~solvencyScore': 0.0,
 'profit_final~profit': 0.0,
 'utilisation_final~utilisation': 0.0,
 'price_final~price': 0.0,
 'churn_final~churnCount': 0.0,
 'capacity_final~capacity': 0.0,
 'incentive_final~incentive': 0.0,
 'retention_final~retention_final_calc': 7.817097663620487e-17,
 'burn_to_mint_final~burn_to_mint_final_calc': 1.4210854715202004e-14}

## 4) Signal Timing Recompute (ONO)
Recompute signal weeks from baseline-relative threshold crossings and compare with `dtse_cross_scenario_signals_ono.csv`.


In [5]:
thr = {row['threshold_id']: row['value'] for _, row in thresholds.iterrows()}

pt = thr['signal_price_drop_threshold_pct']
ut = thr['signal_utilisation_drop_threshold_pct']
prt = thr['signal_profit_drop_threshold_pct']
pvt = thr['signal_providers_drop_threshold_pct']
ct = thr['signal_churn_increase_threshold_pct']

base = med[(med['scenario_id'] == 'baseline_neutral') & (med['profile_id'] == 'ono_v3_calibrated')].sort_values('week')

def first_week(base_series, scen_series, direction, threshold):
    for w, (b, s) in enumerate(zip(base_series, scen_series)):
        if abs(b) < 1e-9:
            continue
        delta = (s - b) / abs(b)
        if direction == 'down' and delta <= -threshold:
            return w
        if direction == 'up' and delta >= threshold:
            return w
    return -1

rows = []
for sid in ['demand_contraction', 'liquidity_shock', 'competitive_yield_pressure', 'provider_cost_inflation']:
    scen = med[(med['scenario_id'] == sid) & (med['profile_id'] == 'ono_v3_calibrated')].sort_values('week')
    rows.append({
        'scenario_id': sid,
        'earliest_price_signal_week': first_week(base['price'].tolist(), scen['price'].tolist(), 'down', pt),
        'earliest_utilisation_signal_week': first_week(base['utilisation'].tolist(), scen['utilisation'].tolist(), 'down', ut),
        'earliest_profit_signal_week': first_week(base['profit'].tolist(), scen['profit'].tolist(), 'down', prt),
        'lagged_provider_signal_week': first_week(base['providers'].tolist(), scen['providers'].tolist(), 'down', pvt),
        'lagged_churn_signal_week': first_week(base['churnCount'].tolist(), scen['churnCount'].tolist(), 'up', ct),
    })

recomputed = pd.DataFrame(rows).sort_values('scenario_id').reset_index(drop=True)
source = signals.sort_values('scenario_id').reset_index(drop=True)

signal_check = {
    'match_exact': recomputed.equals(source),
}

signal_check


{'match_exact': True}

## 5) Interpretation Constraints (Quantified)
Compute two constraints used in thesis interpretation: baseline drift and cross-profile scale spread.


In [6]:
ono_base = med[(med['scenario_id'] == 'baseline_neutral') & (med['profile_id'] == 'ono_v3_calibrated')].sort_values('week')

p0 = float(ono_base[ono_base['week'] == 0]['price'].iloc[0])
pN = float(ono_base[ono_base['week'] == 51]['price'].iloc[0])
pr0 = float(ono_base[ono_base['week'] == 0]['providers'].iloc[0])
prN = float(ono_base[ono_base['week'] == 51]['providers'].iloc[0])

liq = final[final['scenario_id'] == 'liquidity_shock']['solvency_final']

constraints = {
    'ono_baseline_price_change_pct': (pN - p0) / p0,
    'ono_baseline_providers_change_pct': (prN - pr0) / pr0,
    'liquidity_solvency_max': float(liq.max()),
    'liquidity_solvency_min': float(liq.min()),
    'liquidity_solvency_max_min_ratio': float(liq.max() / liq.min()),
}

constraints


{'ono_baseline_price_change_pct': -0.9361817705648581,
 'ono_baseline_providers_change_pct': -0.99,
 'liquidity_solvency_max': 158.33333333333334,
 'liquidity_solvency_min': 1.1261423343107697,
 'liquidity_solvency_max_min_ratio': 140.59797639188986}

## 6) Memo-ready Summary
Produces a compact summary dict for copy-paste into thesis working-memory notes.


In [7]:
summary = {
    'integrity': integrity_report,
    'max_abs_diffs': max_abs_diffs,
    'signal_match': signal_check,
    'constraints': constraints,
    'manifest_key': {
        'scenario_grid_id': manifest['scenario_grid_id'],
        'master_seed': manifest['master_seed'],
        'n_sims': manifest['n_sims'],
        'horizon_weeks': manifest['horizon_weeks'],
        'commit_hash': manifest['dtse_repo']['commit_hash'],
    }
}
summary


{'integrity': {'artifacts_listed': 18,
  'missing_count': 0,
  'hash_mismatch_count': 0},
 'max_abs_diffs': {'providers_final~providers': 0.0,
  'solvency_final~solvencyScore': 0.0,
  'profit_final~profit': 0.0,
  'utilisation_final~utilisation': 0.0,
  'price_final~price': 0.0,
  'churn_final~churnCount': 0.0,
  'capacity_final~capacity': 0.0,
  'incentive_final~incentive': 0.0,
  'retention_final~retention_final_calc': 7.817097663620487e-17,
  'burn_to_mint_final~burn_to_mint_final_calc': 1.4210854715202004e-14},
 'signal_match': {'match_exact': True},
 'constraints': {'ono_baseline_price_change_pct': -0.9361817705648581,
  'ono_baseline_providers_change_pct': -0.99,
  'liquidity_solvency_max': 158.33333333333334,
  'liquidity_solvency_min': 1.1261423343107697,
  'liquidity_solvency_max_min_ratio': 140.59797639188986},
 'manifest_key': {'scenario_grid_id': 'dtse_ch8_grid_v1',
  'master_seed': 20260224,
  'n_sims': 60,
  'horizon_weeks': 52,
  'commit_hash': '0f7b69d0b5776137bddcea6ed